# Git 2.56.1 보안 패치 읽기, 내 Git 점검과 바뀐 동작 확인

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/git_2_56_1_patch.ipynb)

Colab의 기본 Git이 2026년 11월 보안 수정 버전인지 확인합니다. 같은 점검 도구를 내려받아 내 PC에서도 쓸 수 있습니다.
Git 공식 태그 두 개를 비교해 취약점별 커밋 수, 변경 파일 수, 소스와 테스트의 변경 줄 수를 셉니다.
두 버전을 빌드해 임시 저장소에서 가져오기와 인덱스 경로 길이의 동작을 확인합니다.
공격 코드 없이 무해한 입력만 사용합니다. [GitHub 원문 · Git security vulnerabilities announced (2026-11-10)](https://github.blog/open-source/git/git-security-vulnerabilities-announced-7/)

## 1. Colab의 Git 버전 판정

표의 버전은 각 유지보수 계열에서 이번 수정을 처음 포함한 업스트림 릴리스입니다. 배포판이 기존 번호에 수정을 백포트할 수 있으므로 벤더 빌드는 배포판 공지도 확인하세요. 아래 코드는 표의 경계값과 시험 버전 등 23개 예제를 검사한 뒤 Colab의 기본 Git을 조회합니다.

In [ ]:
CHECK_SOURCE = (
    r'''"""11월 보안 릴리스 기준 Git 버전 점검."""
import os
import re
import subprocess
import sys
from pathlib import Path


FIXED_PATCH = {
    50: 2, 51: 3, 52: 1, 53: 1, 54: 1, 55: 1, 56: 1,
}
VENDOR_NOTE = "벤더 빌드, 업스트림 번호로만 판정, 벤더 공지 확인 필요"


def classify(version):
    pattern = r"git version (\d+)\.(\d+)\.(\d+)([.()+\s-].*)?"
    match = re.fullmatch(pattern, version)
    if not match:
        return "unknown", "버전 형식 확인 필요"
    major, minor, patch = map(int, match.group(1, 2, 3))
    suffix = match[4] or ""
    preview = r"(?:^|[. -])(?:rc|alpha|beta|pre|dev)\d*"
    if re.search(preview, suffix, re.I):
        return "unknown", "시험 버전, 정식 릴리스 확인 필요"
    if (major, minor) < (2, 50):
        status = "unsupported"
        note = "지원 계열 아님, 2.56.1 로 올리기"
'''
    "    elif (major, minor) > (2, 56) or "
    "patch >= FIXED_PATCH[minor]:"
    r'''
        status, note = "fixed", "수정됨"
    else:
        status = "unfixed"
        note = f"미수정, 2.{minor}.{FIXED_PATCH[minor]} 이상 필요"
    if suffix:
        note += " · " + VENDOR_NOTE
    return status, note


def inspect_git(path):
    try:
        proc = subprocess.run(
            [path, "--version"], capture_output=True, text=True,
            errors="replace", timeout=10, check=True,
        )
        version = proc.stdout.strip()
        status, note = classify(version)
    except (OSError, subprocess.SubprocessError) as exc:
        version, status = "", "unknown"
        note = f"버전 조회 실패 ({type(exc).__name__})"
    return dict(version=version, status=status, verdict=note)


def installations(extra):
    names = ["git.exe", "git"] if os.name == "nt" else ["git"]
    candidates = []
    for directory in os.get_exec_path():
        for name in names:
            path = Path(directory) / name
            if path.is_file() and os.access(path, os.X_OK):
                candidates.append(path)
    candidates.extend(Path(item).expanduser() for item in extra)
    seen = set()
    for path in candidates:
        key = os.path.normcase(os.path.realpath(path))
        if key not in seen:
            seen.add(key)
            yield path.absolute()


def self_test():
    cases = {
        "unsupported": ("2.49.0", "1.99.9"),
        "unfixed": (
            "2.50.0", "2.50.1", "2.51.2", "2.52.0",
            "2.53.0", "2.54.0 (Apple Git-157)", "2.55.0",
            "2.56.0",
        ),
        "fixed": (
            "2.50.2", "2.51.3", "2.51.10", "2.52.1",
            "2.53.1", "2.54.1", "2.55.1", "2.56.1",
            "2.56.1.windows.1", "2.57.0", "3.0.0",
        ),
        "unknown": ("2.56.1-rc1", "unknown"),
    }
    count = 0
    for expected, versions in cases.items():
        for version in versions:
            status, note = classify("git version " + version)
            vendor = "Apple" in version or "windows" in version
'''
    "            if status != expected or "
    "(VENDOR_NOTE in note) != vendor:"
    r'''
                raise ValueError(f"{version}: {status} {note}")
            count += 1
    print(f"self-test: {count}/{count} 통과")


def main(extra):
    self_test()
    statuses = []
    for i, path in enumerate(installations(extra), 1):
        row = inspect_git(path)
        statuses.append(row["status"])
        print(f"Git {i}", row["version"] or "(조회 실패)",
              row["verdict"], sep=" | ")
    if not statuses:
        print("PATH에서 Git을 찾을 수 없습니다.")
    if any(s in {"unsupported", "unfixed"} for s in statuses):
        return 1
    return 2 if not statuses or "unknown" in statuses else 0


if __name__ == "__main__":
    raise SystemExit(main(sys.argv[1:]))
'''
)
from types import ModuleType

checker = ModuleType("check_git")
exec(compile(CHECK_SOURCE, "check_git.py", "exec"),
     checker.__dict__)
print("유지보수 계열 | 최초 수정 버전")
for minor, patch in checker.FIXED_PATCH.items():
    print(f"2.{minor}.x       | 2.{minor}.{patch}")
checker.self_test()
default_git = checker.inspect_git("git")
print("기본 Git", default_git["version"], default_git["verdict"],
      sep=" | ")

## 2. 내 PC에서 같은 점검 실행하기

이 셀을 실행한 뒤 Colab 왼쪽 파일 목록에서 `check_git.py`를 다운로드하세요. 내 PC에서 `python3 check_git.py`를 실행하면 PATH에 있는 모든 Git을 검사합니다. 추가 실행 파일은 위치 인자로 지정할 수 있습니다. 종료 코드 0은 모두 수정 버전, 1은 미수정 또는 지원 범위 밖 설치본 발견, 2는 조회·판정 불가입니다.

In [ ]:
from pathlib import Path

Path("check_git.py").write_text(CHECK_SOURCE, encoding="utf-8")
print("check_git.py 저장 완료")

## 3. 실행 설정

전체 실행에는 Git, make, C 컴파일러와 zlib 개발 헤더가 필요하며, Rust 코드는 `NO_RUST=1` 로 빼고 빌드합니다. `QUICK = True`이면 소스의 diff 집계와 그래프만 얻습니다. 그래프 언어는 `ko`, `en`, `both` 중에서 고르세요.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
import errno
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
from urllib.request import urlopen

QUICK = False
CHART_LANG = os.environ.get("CHART_LANG", "ko")
if CHART_LANG not in ("ko", "en", "both"):
    raise ValueError("CHART_LANG은 ko, en, both 중 하나입니다.")

OLD, NEW = TAGS = ("v2.56.0", "v2.56.1")
CACHE = Path("cache")
STUDY = CACHE / "git-2-56-1-patch"
SRC = STUDY / "git-src"
TREES = STUDY / "worktrees"
INSTALL = STUDY / "install"
RESULTS = Path("results")
REMOTE = "https://github.com/git/git"
mode = "diff-only" if QUICK else "release"
utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


def check_cache():
    marker = STUDY / ".gitignore"
    contents = "# git-2-56-1-patch\n*\n"
    if CACHE.is_symlink() or STUDY.is_symlink():
        raise ValueError("캐시 경로를 확인하세요.")
    if not STUDY.exists():
        STUDY.mkdir(parents=True)
        marker.write_text(contents, encoding="utf-8")
    if (marker.is_symlink() or not marker.is_file()
            or marker.read_text(encoding="utf-8") != contents):
        raise ValueError("이 노트북이 만든 캐시인지 확인하세요.")


def report_line(message):
    for line in str(message).expandtabs(2).split("\n"):
        text, width = "", 0
        for ch in line:
            if unicodedata.category(ch) == "Cc":
                ch = "?"
            wide = unicodedata.east_asian_width(ch) in "WF"
            cells = 0 if unicodedata.combining(ch) else 1 + wide
            if width + cells > 64:
                print(text)
                text, width = "", 0
            text += ch
            width += cells
        print(text)


check_cache()
RESULTS.mkdir(exist_ok=True)
report_line(f"{mode} · UTC {utc}")

## 4. Git 공식 소스 받기

최초 실행은 공식 저장소를 부분 복제합니다. 두 태그가 가리키는 커밋 해시를 기록하고 이후 집계와 빌드에 같은 해시를 씁니다. [Git 소스 · GPL-2.0](https://github.com/git/git/blob/master/COPYING)

In [ ]:
def run(args, cwd=None, *, data=b"", check=True):
    env = {k: v for k, v in os.environ.items()
           if not k.startswith("GIT_")}
    env.update(
        LC_ALL="C", GIT_CONFIG_NOSYSTEM="1",
        GIT_CONFIG_GLOBAL=os.devnull, GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="core.hooksPath",
        GIT_CONFIG_VALUE_0=os.devnull, GIT_TERMINAL_PROMPT="0",
    )
    proc = subprocess.run(args, cwd=cwd, env=env, input=data,
                          capture_output=True)
    if check and proc.returncode:
        lines = (
            proc.stderr.decode(errors="replace").splitlines()
        )
        reason = lines[-1] if lines else "오류 메시지 없음"
        raise RuntimeError(
            f"{Path(args[0]).name} 실패 · {reason}"
        )
    return proc


def fetch_sources():
    if not SRC.exists():
        run(["git", "clone", "--filter=blob:none",
             "--no-checkout", REMOTE, SRC])
    if not (SRC / ".git").is_dir():
        raise ValueError("소스 복제본을 확인하세요.")
    origin = run(["git", "remote", "get-url", "origin"], SRC)
    if origin.stdout.decode().strip() != REMOTE:
        raise ValueError("공식 저장소의 복제본인지 확인하세요.")
    run(["git", "fetch", "--tags", "origin"], SRC)
    hashes = {}
    for tag in TAGS:
        ref = f"refs/tags/{tag}^{{commit}}"
        proc = run(["git", "rev-parse", "--verify", ref], SRC)
        hashes[tag] = proc.stdout.decode().strip()
        report_line(f"{tag}  {hashes[tag]}")
    return hashes


hashes = fetch_sources()

## 5. 커밋과 변경 파일 집계

이전 태그에는 없고 수정 태그에서 도달할 수 있는 각 커밋의 제목·메시지와 파일별 추가·삭제 줄 수를 읽습니다. 병합 커밋 자체의 diff는 제외하고, 이름 변경은 삭제와 추가로 셉니다. 바이너리 파일의 줄 수는 비워 둡니다.

In [ ]:
def parse_stat(raw):
    files = []
    for record in raw.split(b"\0"):
        if record:
            added, deleted, path = record.split(b"\t", 2)
            files.append(dict(
                path=os.fsdecode(path),
                added=None if added == b"-" else int(added),
                deleted=(
                    None if deleted == b"-" else int(deleted)
                ),
            ))
    return files


def collect():
    revision = f"{hashes[OLD]}..{hashes[NEW]}"
    raw = run(["git", "log", "-z", "--format=%H%x00%s%x00%B",
               revision], SRC).stdout
    fields = raw.removesuffix(b"\0").split(b"\0") if raw else []
    if len(fields) % 3:
        raise ValueError("git log 필드 수 오류")
    rows = []
    for pos in range(0, len(fields), 3):
        sha, title, body = (
            f.decode(errors="replace")
            for f in fields[pos:pos+3]
        )
        stat = run([
            "git", "show", "--format=", "--root",
            "--diff-merges=off", "--no-renames",
            "--no-ext-diff",
            "--no-textconv", "--numstat", "-z", sha,
        ], SRC)
        rows.append((sha, title, body, parse_stat(stat.stdout)))
    return rows


changes = collect()
print(f"커밋 {len(changes)}개")

## 6. CVE별 분류와 집계 근거

메시지의 CVE, 제목 키워드, 소스 경로 키워드 순으로 분류합니다. 여러 후보에 걸치는 커밋은 `review`, 일치하는 후보가 없는 일반 변경은 `unmapped`에 둡니다. 변경 규모를 인용할 때에는 표 아래의 커밋별 근거도 확인하세요.

In [ ]:
RULES = {
    "CVE-2026-114466": ("fast-import",),
    "CVE-2026-114469": ("apply.c",),
    "CVE-2026-105437": ("merge-ll", "xdiff", "merge-ort"),
    "CVE-2026-114467": ("gpg-interface", "tag"),
    "CVE-2026-113408": (
        "connect", "remote", "refs", "fetch-pack", "clone",
    ),
    "CVE-2026-111760": ("read-cache", "unpack-trees", "index"),
}
META = {"release-notes", "tests", "version"}
NOTES = [
    "커밋 메시지의 CVE, 제목 키워드, 경로 키워드 순으로 분류.",
    "제목의 remove_signature는 CVE-2026-114467로 분류.",
    "t/, Documentation/RelNotes/, GIT-VERSION-GEN은 "
    "키워드 검색에서 제외.",
    "복수 후보는 review. 전용 변경은 tests, release-notes, "
    "version, metadata.",
    "일치하는 후보가 없는 일반 변경은 unmapped.",
    "줄 수는 커밋별 추가 + 삭제. t/는 테스트, 나머지는 소스(문서 포함).",
    "병합 자체의 diff는 제외. 파일 수는 범주 내 서로 다른 경로 수.",
    "이름 변경은 삭제와 추가로 계산. 바이너리는 파일 수에만 포함.",
]


def path_kind(path):
    if path.startswith("t/"):
        return "tests"
    if path.startswith("Documentation/RelNotes/"):
        return "release-notes"
    if path == "GIT-VERSION-GEN":
        return "version"
    return "source"


def assign(title, body, files):
    pattern = r"\bCVE-\d{4}-\d{4,}\b"
    message = title + "\n" + body
    hits = sorted(set(re.findall(pattern, message.upper())))
    if hits:
        known = len(hits) == 1 and hits[0] in RULES
        return (
            hits[0] if known else "review",
            "메시지 " + ", ".join(hits),
        )
    if "remove_signature" in title.lower():
        return "CVE-2026-114467", "제목 remove_signature"
    kinds = {path_kind(f["path"]) for f in files}
    matches = defaultdict(list)
    for f in files:
        path = f["path"]
        if path_kind(path) == "source":
            for cve, words in RULES.items():
                if any(word in path for word in words):
                    matches[cve].append(path)
    if matches:
        labels = [f"{cve} ({', '.join(paths)})"
                  for cve, paths in matches.items()]
        key = (next(iter(matches)) if len(matches) == 1
               else "review")
        return key, ", ".join(labels)
    if kinds and kinds <= META:
        key = (next(iter(kinds)) if len(kinds) == 1
               else "metadata")
        return key, "전용 파일"
    return "unmapped", "키워드 없음"


def summarize(rows):
    commits = []
    totals = defaultdict(Counter)
    paths = defaultdict(set)
    for sha, title, body, files in rows:
        key, reason = assign(title, body, files)
        commits.append(dict(commit=sha, title=title,
                            category=key, reason=reason,
                            files=files))
        total = totals[key]
        total["commits"] += 1
        for f in files:
            paths[key].add(f["path"])
            test = path_kind(f["path"]) == "tests"
            kind = "test" if test else "source"
            for col in ("added", "deleted"):
                total[f"{kind}_{col}"] += f[col] or 0
            total["binary_changes"] += int(f["added"] is None)
    columns = ("commits", "source_added", "source_deleted",
               "test_added", "test_deleted", "binary_changes")
    summary = {}
    for key in [*RULES, *sorted(totals.keys() - RULES.keys())]:
        row = {col: totals[key][col] for col in columns}
        summary[key] = dict(row, files=len(paths[key]))
    return commits, summary


commits, summary = summarize(changes)
report_line("분류 규칙")
for note in NOTES:
    report_line(note)
for key, words in RULES.items():
    report_line(f"{key}  {', '.join(words)}")

report_line("\n범주             커밋  파일   소스 + / -    테스트 + / -")
for key, row in summary.items():
    report_line(
        f"{key:16} {row['commits']:4} {row['files']:4} "
        f"{row['source_added']:6}/{row['source_deleted']:<6} "
        f"{row['test_added']:5}/{row['test_deleted']}"
    )
report_line("\n커밋별 근거 (+추가 -삭제, 바이너리는 -)")
for item in commits:
    report_line(f"{item['commit'][:12]}  {item['title']}")
    report_line(f"  {item['category']} · {item['reason']}")
    for f in item["files"]:
        stat = "-"
        if f["added"] is not None:
            stat = f"+{f['added']} -{f['deleted']}"
        report_line(f"  {stat}  {f['path']}")
report_line("\n미매핑 및 복수 후보")
review = [c for c in commits
          if c["category"] in {"unmapped", "review"}]
for item in review:
    report_line(f"{item['commit']}  {item['title']}")
if not review:
    report_line("없음")

## 7. 두 태그 빌드

태그마다 별도 작업 트리에 소스를 펼치고 전용 캐시에 설치합니다. 빌드 기록의 커밋 해시와 실행 파일의 버전이 일치하면 재사용합니다. 표의 시간은 처음 빌드했을 때의 초 단위 경과 시간입니다.

In [ ]:
MAKE_FLAGS = [
    "NO_TCLTK=1", "NO_GETTEXT=1", "NO_PERL=1", "NO_PYTHON=1",
    "NO_CURL=1", "NO_EXPAT=1", "NO_OPENSSL=1", "NO_RUST=1",
]


def build(tag, sha):
    tree = TREES / tag
    prefix = INSTALL / tag
    exe = prefix / "bin" / "git"
    stamp = prefix / "study-build.json"
    expected = f"git version {tag[1:]}"
    if stamp.is_file() and exe.is_file():
        cached = json.loads(stamp.read_text())
        if (cached.get("commit") == sha
                and cached.get("make_flags") == MAKE_FLAGS
                and checker.inspect_git(exe)["version"]
                == expected):
            return dict(cached, status="cached"), exe.resolve()
    if not tree.exists():
        TREES.mkdir(exist_ok=True)
        run(["git", "worktree", "add", "--detach",
             tree.resolve(), sha], SRC)
    head = run(["git", "rev-parse", "HEAD"], tree)
    status = run(["git", "status", "--porcelain",
                  "--untracked-files=normal"], tree)
    if head.stdout.decode().strip() != sha or status.stdout:
        raise ValueError(f"{tag} 작업 트리를 확인하세요.")
    jobs = len(os.sched_getaffinity(0))
    started = time.monotonic()
    run(["make", f"-j{jobs}", *MAKE_FLAGS,
         f"prefix={prefix.resolve()}", "install"], tree)
    elapsed = round(time.monotonic() - started, 3)
    if checker.inspect_git(exe)["version"] != expected:
        raise ValueError(f"{tag} 설치본 버전 확인 실패")
    record = dict(commit=sha, seconds=elapsed,
                  make_flags=MAKE_FLAGS)
    stamp.write_text(json.dumps(record), encoding="utf-8")
    return dict(record, status="built"), exe.resolve()


builds = {tag: dict(status="skipped", seconds=None)
          for tag in TAGS}
bins = {}
if not QUICK:
    report_line("빌드 옵션 · " + " ".join(MAKE_FLAGS))
for tag in TAGS:
    if QUICK:
        report_line(f"{tag}  빌드 미측정")
    else:
        print(f"{tag} 빌드 확인 중", flush=True)
        try:
            builds[tag], bins[tag] = build(tag, hashes[tag])
        except (OSError, RuntimeError, ValueError) as exc:
            builds[tag] = dict(status="failed", seconds=None,
                               error=str(exc))
            report_line(f"{tag} 빌드 실패 · {exc}")
            continue
        row = builds[tag]
        report_line(
            f"{tag}  빌드 {row['seconds']}초 ({row['status']})"
        )
missing_tags = ", ".join(tag for tag in TAGS if tag not in bins)

## 8. 동작 A · fast-import의 파일 출력

일반 스트림으로 작은 커밋 하나를 가져옵니다. 두 번째 대조군에서는 호출자가 명령행에서 `edges.txt` 출력을 지정합니다. 각 행은 독립된 임시 저장소에서 측정하며, 표에는 종료 코드, 파일 생성 여부, stderr 첫 줄이 남습니다.

**공지에서 확인한 변경** · 2.56.0은 입력 스트림이 요청한 일부 파일 접근을 허용했고, 2.56.1은 이를 기본 거부합니다. [GitHub 보안 공지 · CVE-2026-114466](https://github.blog/open-source/git/git-security-vulnerabilities-announced-7/)

`stream` 행은 업스트림 `t9300` 테스트와 같은 `option git` 지시를 입력 첫 줄에 넣어 수정판의 기본 거부를 확인합니다. `--allow-unsafe-features`가 없으면 오류를 내고 파일을 만들지 않으며, 있으면 성공해 `edges.txt`를 생성할 것으로 예상합니다. [Git 업스트림 테스트 · t9300](https://code.googlesource.com/git/+/ce5664b1148026fb61463becffe610846ec50411%5E1..ce5664b1148026fb61463becffe610846ec50411/)

In [ ]:
def init_repo(exe, repo):
    run([exe, "init", "-q", "--template=", "."], repo)


def import_control(exe, form, opt_in=False):
    stream = (b"commit refs/heads/sample\n"
              b"committer Sample <> 1000000000 +0000\n"
              b"data 1\nx\n\ndone\n")
    with tempfile.TemporaryDirectory(dir=STUDY) as name:
        repo = Path(name)
        init_repo(exe, repo)
        args = [exe, "fast-import", "--quiet"]
        if form == "cli":
            args.append("--export-pack-edges=edges.txt")
        elif form == "stream":
            stream = (
                b"option git export-pack-edges=edges.txt\n"
                + stream
            )
            if opt_in:
                args.append("--allow-unsafe-features")
        proc = run(args, repo, data=stream, check=False)
        lines = (
            proc.stderr.decode(errors="replace").splitlines()
        )
        return dict(
            form=form, allow_unsafe_features=opt_in,
            returncode=proc.returncode,
            stderr_first=lines[0] if lines else "",
            edges_created=(repo / "edges.txt").is_file(),
        )


behavior = None
if QUICK:
    report_line("A · 동작 실험 미실행")
elif missing_tags:
    print(f"A · 빌드 실패로 건너뜀 ({missing_tags})")
else:
    rows = []
    report_line("A · 형식 / 허용 옵션 / 종료 코드 / 파일 생성 / stderr 첫 줄")
    for tag in TAGS:
        cases = [("ordinary", False), ("cli", False)]
        if tag == NEW:
            cases += [("stream", False), ("stream", True)]
        for form, opt_in in cases:
            trial = import_control(bins[tag], form, opt_in)
            row = dict(tag=tag, **trial)
            rows.append(row)
            report_line(
                f"{tag} {form} {row['allow_unsafe_features']} "
                f"{row['returncode']} "
                f"{row['edges_created']}"
            )
            report_line(
                "  " + (row["stderr_first"] or "(stderr 없음)")
            )
    behavior = {"fast_import": {
        "trials": rows, "old_protected_stream": "미실행",
    }}

## 9. 동작 B · 수정판이 수락하는 인덱스 경로 길이

수정판에서 경로 길이를 두 배씩 늘리고 첫 거부 구간을 이진 탐색합니다. 경로는 인덱스에만 추가하고 작업 트리에 펼치지 않습니다. 운영체제의 인수 길이 제한이 먼저 나타나면 Git 자체의 한도는 미확정으로 남깁니다.

`index.maxPathLen`의 기본값은 1073740799바이트로, Colab에서는 운영체제의 인수 길이 제한 때문에 이 값에 도달할 수 없습니다. 아래 설정 확인 행은 한도를 4096바이트로 낮춰 수정판이 4096바이트를 수락하고 4097바이트를 거부하는지 확인합니다.

이전 버전은 지정된 네 길이 중 수정판이 수락한 길이만 대조합니다. 매 시도에 빈 임시 저장소를 사용합니다.

In [ ]:
def index_trial(exe, length, *, max_path_len=None):
    leaf = "f" if length % 2 else "ff"
    path = "a/" * ((length - 1) // 2) + leaf
    with tempfile.TemporaryDirectory(dir=STUDY) as repo:
        init_repo(exe, repo)
        blob = run([exe, "hash-object", "-w", "--stdin"], repo)
        oid = blob.stdout.decode().strip()
        args = [exe]
        if max_path_len is not None:
            args += ["-c", f"index.maxPathLen={max_path_len}"]
        args += ["update-index", "--add", "--cacheinfo",
                 f"100644,{oid},{path}"]
        try:
            proc = run(args, repo, check=False)
        except OSError as exc:
            if exc.errno != errno.E2BIG:
                raise
            return dict(bytes=length, outcome="os-argv-limit",
                        returncode=None, stderr_first="")
    msg = proc.stderr.decode(errors="replace")
    lines = msg.replace(path, "<path>").splitlines()
    pattern = (
        r"(?:name|path).*too long"
        r"|path.*(?:length|longer).*exceed"
    )
    if proc.returncode == 0:
        outcome = "accepted"
    elif max_path_len is not None:
        outcome = "rejected"
    elif any("file name too long" in s.lower() for s in lines):
        outcome = "os-path-limit"
    elif any(re.search(pattern, s, re.I) for s in lines):
        outcome = "length-rejected"
    else:
        outcome = "other-error"
    return dict(bytes=length, outcome=outcome,
                returncode=proc.returncode,
                stderr_first=lines[0] if lines else "")


def index_checks(bins):
    if checker.inspect_git(
        bins[NEW]
    )["version"] != "git version 2.56.1":
        raise ValueError("길이 탐색에 필요한 수정판 버전 확인 실패")
    trials = []

    def measure(n):
        row = index_trial(bins[NEW], n)
        trials.append(row)
        return row["outcome"]

    low, high, ceiling = 0, 1, 262144
    limits = {"os-argv-limit", "length-rejected"}
    while True:
        result = measure(high)
        if result != "accepted" or high == ceiling:
            break
        low, high = high, min(high * 2, ceiling)
    if result in limits:
        while high - low > 1:
            mid = (low + high) // 2
            state = measure(mid)
            if state == "accepted":
                low = mid
            elif state in limits:
                high, result = mid, state
            else:
                result = state
                break
    elif result == "accepted":
        low = high
    controls = []
    for size in (4095, 65535, 65536, 100000):
        fixed = next((r for r in trials if r["bytes"] == size),
                     None)
        state = fixed["outcome"] if fixed else measure(size)
        accepted = state == "accepted"
        code = None
        if accepted:
            code = index_trial(bins[OLD], size)["returncode"]
        controls.append(dict(bytes=size, run=accepted,
                             returncode=code))
    observed = (r["bytes"] for r in trials
                if r["outcome"] == "accepted")
    known = low and result == "length-rejected"
    configured = [
        dict(max_path_len=limit, **index_trial(
            bins[NEW], size, max_path_len=limit))
        for limit, size in ((4096, 4096), (4096, 4097),
                            (2147483648, 16))
    ]
    overflow = configured[-1]
    overflow["config_rejected"] = (
        overflow["outcome"] == "rejected"
        and "value for 'index.maxpathlen' exceeds maximum"
        in overflow["stderr_first"]
    )
    return dict(old_controls=controls, new_trials=trials,
                largest_observed_accepted=max(observed,
                                              default=None),
                boundary_status=result,
                max_accepted_bytes=low if known else None,
                max_path_len_config=configured)


if QUICK:
    report_line("B · 동작 실험 미실행")
elif missing_tags:
    print(f"B · 빌드 실패로 건너뜀 ({missing_tags})")
else:
    index = behavior["index"] = index_checks(bins)
    report_line("B · 바이트 / 판정 / 종료 코드 / stderr 첫 줄")
    for row in index["old_controls"]:
        value = (row["returncode"] if row["run"]
                 else "미실행 (수정판 미수락)")
        report_line(f"{OLD} {row['bytes']}  {value}")
    for row in index["new_trials"]:
        report_line(f"{NEW} {row['bytes']} {row['outcome']} "
                    f"{row['returncode']}")
        if row["stderr_first"]:
            report_line("  " + row["stderr_first"])
    report_line(
        f"최대 수락 길이  {index['max_accepted_bytes'] or '미확정'}"
    )
    observed = index["largest_observed_accepted"]
    report_line(f"관측된 수락 하한  {observed} 바이트")
    report_line("탐색 종료 사유  " + index["boundary_status"])
    report_line("B · index.maxPathLen / 바이트 / 판정 / 종료 코드"
                " / stderr 첫 줄")
    report_line(f"{OLD} 미실행 (index.maxPathLen 설정 없음)")
    for row in index["max_path_len_config"]:
        report_line(
            f"{NEW} {row['max_path_len']} {row['bytes']} "
            f"{row['outcome']} {row['returncode']}"
        )
        stderr = row["stderr_first"] or "(stderr 없음)"
        report_line("  " + stderr)

## 10. 한글 글꼴 준비

한국어 그래프를 그릴 때 NanumGothic TTF를 `cache/fonts/`에 내려받아 Matplotlib에 등록합니다. [Google Fonts · NanumGothic, SIL Open Font License 1.1](https://github.com/google/fonts/tree/main/ofl/nanumgothic)

In [ ]:
os.environ.setdefault("MPLCONFIGDIR", str(CACHE / "matplotlib"))
from matplotlib import font_manager
import matplotlib.pyplot as plt

if CHART_LANG in ("ko", "both"):
    font = CACHE / "fonts" / "NanumGothic-Regular.ttf"
    try:
        if not font.exists():
            url = (
                "https://raw.githubusercontent.com/google/fonts"
                "/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
            )
            with urlopen(url, timeout=30) as res:
                content = res.read()
            font.parent.mkdir(exist_ok=True)
            font.write_bytes(content)
        font_manager.fontManager.addfont(font)
        props = font_manager.FontProperties(fname=font)
        name = props.get_name()
        font_manager.findfont(
            font_manager.FontProperties(family=name),
            fallback_to_default=False,
        )
        ko_font = name
    except (OSError, RuntimeError, ValueError) as error:
        print(
            "경고: NanumGothic 사용 불가 "
            f"({type(error).__name__}). 시스템 기본 글꼴 사용.",
            file=sys.stderr,
        )
        ko_font = plt.rcParamsDefault["font.family"]

## 11. 취약점별 변경 줄 수

가로 막대는 해당 CVE로 분류된 커밋의 추가·삭제 줄 수 합계입니다. `t/`는 테스트, 문서를 포함한 나머지는 소스로 집계합니다. `both`는 한국어와 영어 그림을 차례로 표시하고 각각 PNG로 저장합니다.

In [ ]:
charts = []
languages = (("ko", "en") if CHART_LANG == "both"
             else (CHART_LANG,))
for lang in languages:
    ko = lang == "ko"
    family = ko_font if ko else "DejaVu Sans"
    xlabel = ("바뀐 줄 수 (추가 + 삭제)" if ko
              else "Changed lines (additions + deletions)")
    with plt.rc_context({"font.family": family}):
        fig, ax = plt.subplots(figsize=(8, 4.1),
                               layout="constrained")
        left = [0] * len(RULES)
        series = [
            ("source", "소스" if ko else "Source", "#477688"),
            ("test", "테스트" if ko else "Tests", "#c9a66e"),
        ]
        for kind, label, color in series:
            counts = [summary[cve][kind + "_added"]
                      + summary[cve][kind + "_deleted"]
                      for cve in RULES]
            ax.barh(list(RULES), counts, left=left,
                    label=label, color=color)
            left = counts
        ax.set_xlabel(xlabel)
        ax.set_ylabel("취약점" if ko else "CVE")
        ax.invert_yaxis()
        ax.set_axisbelow(True)
        ax.grid(axis="x", alpha=0.2)
        ax.legend()
        filename = f"fig1_patch_lines_by_cve_{lang}.png"
        fig.savefig(RESULTS / filename, dpi=150)
        charts.append(filename)
        plt.show()
        plt.close(fig)
        print(RESULTS / filename)

## 12. 결과 저장과 정리

`results/results.json`에는 태그 해시, 커밋별 분류 근거, 표 수치, 동작 측정값, 이번 실행의 PNG 파일명이 들어갑니다. 기본값인 `CLEAN = True`는 전용 캐시의 소스·작업 트리·설치본을 지웁니다. 같은 빌드를 다시 쓰려면 아래 값을 `False`로 바꾸세요.

In [ ]:
CLEAN = True
data = dict(utc=utc, mode=mode, tags=hashes, builds=builds,
            default_git=default_git, rules=RULES,
            methodology=NOTES, commits=commits, summary=summary,
            behavior=behavior, charts=charts)
out = RESULTS / "results.json"
pending = out.with_suffix(".json.tmp")
pending.write_text(
    json.dumps(data, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
pending.replace(out)
print(out)
if CLEAN:
    check_cache()
    for directory in (TREES, INSTALL, SRC):
        if directory.exists():
            shutil.rmtree(directory)
    print("소스·작업 트리·설치본 정리 완료")
free = shutil.disk_usage(CACHE).free
print(f"남은 디스크 공간 {free / 2**30:.2f} GiB")